# Vector Search in Embeddings Databases

This notebook finds audio windows that **sound similar to your example clips** ("templates"), by comparing **foundation-model embeddings** — no trained classifier required.

---

### What this notebook does (in order):
1. **Connects to your Google Drive** to read audio and save results
2. **Installs the necessary software** automatically
3. **Computes a template embedding** for each example audio clip and saves it as a `.npy` file next to the clip *(skipped if it already exists)*
4. **Runs a vector similarity search** of every template against one or more pre-built **embeddings databases** and writes the matches to CSV
5. **Extracts an audio segment** for each match, organised into one folder per site and label
6. **Reviews the extracted segments** one by one in an interactive spectrogram + audio player, marking each as *true* or *false*

### Supported preset models:
| Preset | HuggingFace repo | Sample rate | Window | Embedding size |
|---|---|---|---|---|
| Perch v2.0 | `biodiversica/Perch-onnx-backbone` | 32 000 Hz | 5.0 s | 1536 |
| BirdNET v2.4 | `biodiversica/BirdNET-onnx-backbone` | 48 000 Hz | 3.0 s | 1024 |

> The backbone you select here **must match** the one used to build the embeddings database you search against (same sample rate, window, and embedding size).

### Templates:
Point this notebook at a Google Drive folder of short example audio clips. **Each clip's filename (without extension) becomes its label.** To provide several examples for one label, group the clips in a subfolder — the subfolder name becomes the label and all of its clips are combined.

### Embeddings databases:
Search against a single `*.embeddings.db` file, or a folder containing several of them, produced by the **Compute Embeddings Database** notebook.

### Output:
For each database, a results CSV is written to your output folder with columns
`label, site, date, time, similarity_score, start_time, end_time` (the `similarity_score` column holds the similarity score). Steps 5 and 6 then extract and let you review the matching audio clips — all within this notebook, no other notebook required.

### How to run:
Run each cell **one at a time**, top to bottom. Click ▶ or press `Shift + Enter`.

> **New to notebooks?** A cell with `[ ]` on the left has not been run. After running it shows `[1]`, `[2]`, etc. If you see an error (red text), read the message — it usually tells you exactly what to fix.

---

Created by [biodiversica](https://biodiversica.xyz). For issues, questions, or feedback, open an issue on [GitHub](https://github.com/biodiversica/bioacoustic-ipynbs/issues) or visit [biodiversica.xyz](https://biodiversica.xyz).

---
## Step 1 — Connect Google Drive & Install Software

Run the two cells below. The first will ask you to **allow access to your Google Drive** — click the link and follow the steps.

In [ ]:
#@title 📂 Connect Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive connected successfully.')

In [ ]:
#@title 📦 Install packages { display-mode: "form" }

!pip install onnxruntime librosa soundfile huggingface_hub pandas matplotlib ipywidgets -q

print('\nAll packages installed successfully.')
print('(The Arbimon SDK is installed later, only if you use Arbimon as the audio source in Step 5.)')

---
## Step 2 — Configuration

Fill in the forms below. **You do not need to edit any code** — just type or select your values in each form and run the cell.

Run all three forms from top to bottom:
1. **General Settings** — template folder, databases path, output folder
2. **Model** — which backbone to use (must match the databases you search)
3. **Search Settings** — how many matches to keep and how to score them

> **Tip:** The forms hide the code automatically. To see the underlying code, click the `{ }` icon in the top-right corner of any form cell.

In [52]:
#@title ⚙️ General Settings { display-mode: "form" }

import os

#@markdown **Audio templates folder** — Google Drive folder with your example clips.
#@markdown Each clip's filename becomes its label; group clips in a subfolder to combine several
#@markdown examples under one label (the subfolder name). Computed `.npy` embeddings are saved here.
DRIVE_TEMPLATES_DIR = '/content/drive/MyDrive/Templates'  #@param {type:"string"}

#@markdown ---
#@markdown **Embeddings databases** — full path to a single `*.embeddings.db` file,
#@markdown **or** to a folder containing one or more of them (searched recursively).
DRIVE_EMBEDDINGS_DB_PATH = '/content/drive/MyDrive/Embeddings'  #@param {type:"string"}

#@markdown ---
#@markdown **Results output folder** — where the search result CSV files will be saved on your Google Drive.
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/vector_search_results'  #@param {type:"string"}

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

print(f'Templates folder : {DRIVE_TEMPLATES_DIR}')
if not os.path.isdir(DRIVE_TEMPLATES_DIR):
    print('  WARNING: folder not found — check the path and that Drive is connected.')
print(f'Databases path   : {DRIVE_EMBEDDINGS_DB_PATH}')
if not os.path.exists(DRIVE_EMBEDDINGS_DB_PATH):
    print('  WARNING: path not found — check the path and that Drive is connected.')
print(f'Results folder   : {DRIVE_RESULTS_DIR}')

In [53]:
#@title 🤖 Model { display-mode: "form" }

import os

#@markdown **Model preset** — select the foundation model backbone.
#@markdown This **must match** the backbone used to build the embeddings database(s) you search.
#@markdown - **Perch v2**: Google Perch backbone (32 kHz · 5 s window · 1536-dim embeddings)
#@markdown - **BirdNET backbone**: BirdNET v2.4 backbone (48 kHz · 3 s window · 1024-dim embeddings)
#@markdown - **Custom ONNX**: provide your own ONNX backbone model
MODEL_PRESET = 'BirdNET backbone'  #@param ["Perch v2", "BirdNET backbone", "Custom ONNX"]

#@markdown ---
#@markdown ### Custom ONNX settings *(used when preset = Custom ONNX)*

#@markdown **Custom model access**
CUSTOM_MODEL_SOURCE = 'google_drive'  #@param ["google_drive", "huggingface"]

#@markdown **Google Drive path** — full path to your `.onnx` backbone file on Drive.
CUSTOM_DRIVE_MODEL_PATH = '/content/drive/MyDrive/models/my_backbone.onnx'  #@param {type:"string"}

#@markdown **HuggingFace repo ID** — e.g. `my_org/my-backbone-onnx`
CUSTOM_HF_REPO = ''  #@param {type:"string"}

#@markdown **HuggingFace filename** — e.g. `backbone.onnx`
CUSTOM_HF_FILE = 'backbone.onnx'  #@param {type:"string"}

#@markdown **Sample rate (Hz)**
CUSTOM_SAMPLE_RATE = 48000  #@param {type:"integer"}

#@markdown **Window duration (seconds)** — length of each audio window fed to the model.
CUSTOM_WINDOW_S = 3.0  #@param {type:"number"}

#@markdown **ONNX input node name**
CUSTOM_INPUT_NAME = 'input'  #@param {type:"string"}

#@markdown **ONNX output node name**
CUSTOM_OUTPUT_NAME = 'embedding'  #@param {type:"string"}

#@markdown **Embedding size** — number of dimensions in the output vector.
CUSTOM_EMBEDDING_SIZE = 1024  #@param {type:"integer"}

_PRESETS = {
    'Perch v2': {
        'hf_repo':        'biodiversica/Perch-onnx-backbone',
        'hf_file':        'perch_v2_backbone.onnx',
        'sample_rate':    32000,
        'window_s':       5.0,
        'input_name':     'inputs',
        'output_name':    'embedding',
        'embedding_size': 1536,
        'model_name':     'perch_v2_backbone',
    },
    'BirdNET backbone': {
        'hf_repo':        'biodiversica/BirdNET-onnx-backbone',
        'hf_file':        'model_backbone.onnx',
        'sample_rate':    48000,
        'window_s':       3.0,
        'input_name':     'INPUT',
        'output_name':    'embedding',
        'embedding_size': 1024,
        'model_name':     'model_backbone',
    },
}

if MODEL_PRESET in _PRESETS:
    _p = _PRESETS[MODEL_PRESET]
    HF_REPO        = _p['hf_repo']
    HF_FILE        = _p['hf_file']
    SAMPLE_RATE    = _p['sample_rate']
    WINDOW_S       = _p['window_s']
    INPUT_NAME     = _p['input_name']
    OUTPUT_NAME    = _p['output_name']
    EMBEDDING_SIZE = _p['embedding_size']
    MODEL_NAME     = _p['model_name']
else:
    SAMPLE_RATE    = CUSTOM_SAMPLE_RATE
    WINDOW_S       = CUSTOM_WINDOW_S
    INPUT_NAME     = CUSTOM_INPUT_NAME
    OUTPUT_NAME    = CUSTOM_OUTPUT_NAME
    EMBEDDING_SIZE = CUSTOM_EMBEDDING_SIZE
    if CUSTOM_MODEL_SOURCE == 'huggingface':
        HF_REPO    = CUSTOM_HF_REPO.strip()
        HF_FILE    = CUSTOM_HF_FILE.strip()
        MODEL_NAME = os.path.splitext(HF_FILE)[0]
    else:
        HF_REPO    = None
        HF_FILE    = None
        MODEL_NAME = os.path.splitext(os.path.basename(CUSTOM_DRIVE_MODEL_PATH))[0]

WINDOW_SAMPLES = int(WINDOW_S * SAMPLE_RATE)

print(f'Model preset    : {MODEL_PRESET}')
print(f'Model name      : {MODEL_NAME}')
print(f'Sample rate     : {SAMPLE_RATE} Hz')
print(f'Window duration : {WINDOW_S} s  ({WINDOW_SAMPLES} samples)')
print(f'Embedding size  : {EMBEDDING_SIZE}')
if HF_REPO:
    print(f'HF repo         : {HF_REPO}')
    print(f'HF file         : {HF_FILE}')

In [54]:
#@title 🔎 Search Settings { display-mode: "form" }

#@markdown **Top-K** — how many of the best-matching windows to keep.
#@markdown In *per_label* mode this is per label; in *global* mode it is across all labels combined.
TOP_K = 10  #@param {type:"integer"}

#@markdown **Metric** — how window similarity is measured (all oriented so higher = more similar).
METRIC = 'cosine'  #@param ["cosine", "dot", "euclidean"]

#@markdown **Mode**
#@markdown - **per_label** — keep the Top-K windows for *each* label independently
#@markdown - **global** — keep the Top-K windows across *all* labels combined
MODE = 'per_label'  #@param ["per_label", "global"]

#@markdown **Score threshold** — minimum similarity score to keep a window (0.0 keeps all Top-K).
SCORE_THRESHOLD = 0.7  #@param {type:"number"}

#@markdown **Batch size** — number of database windows scored at once (controls memory use).
BATCH_SIZE = 10000  #@param {type:"integer"}

#@markdown ---
#@markdown **Template window overlap (0.0–0.9)** — when an example clip is longer than one model
#@markdown window, this sets the overlap between the windows extracted from it. More overlap = more
#@markdown example vectors per label.
TEMPLATE_OVERLAP = 0.0  #@param {type:"number"}
TEMPLATE_OVERLAP = min(max(TEMPLATE_OVERLAP, 0.0), 0.9)  # keep within 0.0–0.9

print(f'Top-K           : {TOP_K}')
print(f'Metric          : {METRIC}')
print(f'Mode            : {MODE}')
print(f'Score threshold : {SCORE_THRESHOLD}')
print(f'Batch size      : {BATCH_SIZE}')
print(f'Template overlap: {TEMPLATE_OVERLAP}')

---
## Step 3 — Compute Template Embeddings

This cell loads the backbone model and computes an embedding for each example audio clip in your
templates folder. The embedding is saved as a `.npy` file **next to the clip** (same name).

- If a clip is longer than one model window, every window is embedded and stored — each becomes an
  independent example for that label (a database window is then scored against the **best-matching**
  example, i.e. nearest-example).
- If a `.npy` file already exists for a clip, it is **skipped** so you can safely re-run.

**Labels:** a clip placed directly in the templates folder is labelled by its filename. A clip inside
a subfolder is labelled by the **subfolder name**, so several clips in the same subfolder share one label.

In [55]:
#@title 🧠 Load model & compute template embeddings { display-mode: "form" }

import os
import glob as _glob
import numpy as np
import librosa

# --- Resolve and load ONNX model ---
if HF_REPO:
    from huggingface_hub import hf_hub_download
    print(f'Downloading backbone from HuggingFace: {HF_REPO} / {HF_FILE} ...')
    _model_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE)
else:
    _model_path = CUSTOM_DRIVE_MODEL_PATH

if not os.path.exists(_model_path):
    raise FileNotFoundError(f'Model not found: {_model_path}\nCheck your model configuration in Step 2.')

import onnxruntime as ort
_ort_session = ort.InferenceSession(_model_path)
_out_names   = [o.name for o in _ort_session.get_outputs()]
_fetch_output = OUTPUT_NAME if OUTPUT_NAME in _out_names else _out_names[0]
if _fetch_output != OUTPUT_NAME:
    print(f"WARNING: output '{OUTPUT_NAME}' not found in model; using '{_fetch_output}' instead.")

print('ONNX model loaded.')
print(f'  Inputs   : {[i.name for i in _ort_session.get_inputs()]}  →  using \'{INPUT_NAME}\'')
print(f'  Outputs  : {_out_names}  →  fetching \'{_fetch_output}\'')
print(f'  Providers: {_ort_session.get_providers()}')

_template_stride = max(1, int(WINDOW_SAMPLES * (1 - TEMPLATE_OVERLAP)))

def _embed_window(audio_window):
    seg = audio_window.astype(np.float32)
    if len(seg) < WINDOW_SAMPLES:
        seg = np.pad(seg, (0, WINDOW_SAMPLES - len(seg)))
    out = _ort_session.run([_fetch_output], {INPUT_NAME: seg[np.newaxis, :]})[0]
    return out.squeeze(0).astype(np.float32)

def _embed_clip(audio):
    vecs = []
    for start in range(0, max(1, len(audio)), _template_stride):
        seg = audio[start:start + WINDOW_SAMPLES]
        if len(seg) < WINDOW_SAMPLES * 0.5 and vecs:
            continue
        vecs.append(_embed_window(seg))
    return np.stack(vecs) if vecs else None

def _label_for(audio_path, templates_dir):
    parent = os.path.normpath(os.path.dirname(audio_path))
    root   = os.path.normpath(templates_dir)
    if parent != root:
        # clip lives in a subfolder → label is the subfolder name
        return os.path.basename(parent)
    return os.path.splitext(os.path.basename(audio_path))[0]

if not os.path.isdir(DRIVE_TEMPLATES_DIR):
    raise FileNotFoundError(f'Templates folder not found: {DRIVE_TEMPLATES_DIR}')

_clips = sorted(
    _glob.glob(os.path.join(DRIVE_TEMPLATES_DIR, '**', '*.wav'),  recursive=True) +
    _glob.glob(os.path.join(DRIVE_TEMPLATES_DIR, '**', '*.flac'), recursive=True) +
    _glob.glob(os.path.join(DRIVE_TEMPLATES_DIR, '**', '*.mp3'),  recursive=True)
)

if not _clips:
    raise FileNotFoundError(f'No audio clips (.wav/.flac/.mp3) found in: {DRIVE_TEMPLATES_DIR}')

print(f'\nFound {len(_clips)} example clip(s). Embedding dimension: {EMBEDDING_SIZE}')
print('=' * 65)

_computed = 0
_skipped  = 0
for _clip in _clips:
    _npy = os.path.splitext(_clip)[0] + '.npy'
    _label = _label_for(_clip, DRIVE_TEMPLATES_DIR)
    if os.path.exists(_npy):
        print(f'  [skip] {os.path.relpath(_clip, DRIVE_TEMPLATES_DIR)}  (label: {_label})')
        _skipped += 1
        continue
    try:
        _audio, _ = librosa.load(_clip, sr=SAMPLE_RATE, mono=True)
    except Exception as e:
        print(f'  ERROR loading {os.path.basename(_clip)}: {e} — skipping.')
        continue
    _emb = _embed_clip(_audio)
    if _emb is None:
        print(f'  WARNING: {os.path.basename(_clip)} too short to embed — skipping.')
        continue
    np.save(_npy, _emb)
    print(f'  [done] {os.path.relpath(_clip, DRIVE_TEMPLATES_DIR)}  (label: {_label}, {_emb.shape[0]} window(s))')
    _computed += 1

print('=' * 65)
print(f'Template embeddings — computed: {_computed}, skipped (existing): {_skipped}')
print('Continue to Step 4 to run the search.')

---
## Step 4 — Run Vector Search

This cell loads every template `.npy`, then for each embeddings database it scores every stored window
against the templates and keeps the best matches (Top-K) according to your search settings.

For each database, a results CSV is written to your output folder named `<database>.search.csv` with columns
`label, site, date, time, similarity_score, start_time, end_time` (plus `recording_id` for reference).

The `similarity_score` column holds the similarity score. Continue to **Step 5** to extract the matching audio
segments and **Step 6** to review them — directly in this notebook.

In [56]:
#@title 🚀 Run vector search { display-mode: "form" }

import os
import csv
import glob as _glob
import heapq
import sqlite3
from datetime import datetime
from pathlib import Path
import numpy as np

# ── Load templates from the .npy files saved in Step 3 ───────────────────────
_template_npys = sorted(_glob.glob(os.path.join(DRIVE_TEMPLATES_DIR, '**', '*.npy'), recursive=True))
if not _template_npys:
    raise FileNotFoundError('No template .npy files found. Run Step 3 first.')

templates = {}  # label -> (n_examples, dim) float32
for _npy in _template_npys:
    _arr = np.load(_npy).astype(np.float32)
    if _arr.ndim == 1:
        _arr = _arr[np.newaxis, :]
    _label = _label_for(_npy, DRIVE_TEMPLATES_DIR)
    templates.setdefault(_label, []).append(_arr)
templates = {lbl: np.vstack(parts) for lbl, parts in templates.items()}

_label_list = list(templates.keys())
_template_dim = next(iter(templates.values())).shape[1]
_dims = {a.shape[1] for a in templates.values()}
if len(_dims) > 1:
    raise ValueError(f'Templates have inconsistent dimensions: {_dims}')

# Flat template matrix + parallel label list, and per-label column indices.
_flat_templates = np.vstack([templates[l] for l in _label_list])
_template_labels = [l for l in _label_list for _ in range(templates[l].shape[0])]
_label_cols = {}
for _ci, _lbl in enumerate(_template_labels):
    _label_cols.setdefault(_lbl, []).append(_ci)
_label_cols = {l: np.array(idxs) for l, idxs in _label_cols.items()}

print(f'Loaded {sum(t.shape[0] for t in templates.values())} template vector(s) '
      f'across {len(_label_list)} label(s): {_label_list}')
print(f'Template dimension: {_template_dim}')

# ── Scoring helpers (cosine / dot / euclidean) ───────────────────────────────
def _compute_scores(batch, tmpl, metric):
    if metric == 'cosine':
        b = batch / (np.linalg.norm(batch, axis=1, keepdims=True) + 1e-8)
        t = tmpl / (np.linalg.norm(tmpl, axis=1, keepdims=True) + 1e-8)
        return (b @ t.T).astype(np.float32)
    if metric == 'dot':
        return (batch @ tmpl.T).astype(np.float32)
    if metric == 'euclidean':
        b_sq = np.sum(batch ** 2, axis=1, keepdims=True)
        t_sq = np.sum(tmpl ** 2, axis=1)
        dist_sq = b_sq + t_sq - 2.0 * (batch @ tmpl.T)
        return (-np.sqrt(np.maximum(dist_sq, 0.0))).astype(np.float32)
    raise ValueError(f"Unknown metric '{metric}'.")

def _iter_embeddings_batched(db_path, batch_size):
    conn = sqlite3.connect(db_path)
    try:
        cur = conn.cursor()
        cur.execute('SELECT site_name, recording_id, chunk_index, datetime, data '
                    'FROM embeddings ORDER BY site_name, recording_id, chunk_index')
        while True:
            rows = cur.fetchmany(batch_size)
            if not rows:
                break
            keys = [(r[0], r[1], r[2], r[3]) for r in rows]
            matrix = np.stack([np.frombuffer(r[4], dtype=np.float32).copy() for r in rows])
            yield keys, matrix
    finally:
        conn.close()

def _load_db_metadata(db_path):
    conn = sqlite3.connect(db_path)
    try:
        return dict(conn.execute('SELECT key, value FROM metadata').fetchall())
    finally:
        conn.close()

def _chunk_times(chunk_index, window_s, sample_rate, overlap):
    stride = max(1, int(window_s * sample_rate * (1.0 - overlap)))
    start = chunk_index * stride / sample_rate
    return start, start + window_s

_RESULT_COLS = ['label', 'site', 'date', 'time', 'similarity_score',
                'start_time', 'end_time', 'recording_id']

def _row_from_key(key, label, score, window_s, sample_rate, overlap):
    site_name, recording_id, chunk_index, dt = key
    start_t, end_t = _chunk_times(chunk_index, window_s, sample_rate, overlap)
    date_str, time_str = '', ''
    iso = dt or ''
    if not iso:
        # try to parse a timestamp from the recording id (e.g. SITE_20250628_185902)
        import re as _re
        m = _re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', recording_id) or \
            _re.search(r'(\d{8})_(\d{6})', recording_id)
        if m:
            g = m.groups()
            if len(g) == 6:
                iso = f'{g[0]}-{g[1]}-{g[2]}T{g[3]}:{g[4]}:{g[5]}'
            else:
                d, t = g
                iso = f'{d[:4]}-{d[4:6]}-{d[6:8]}T{t[:2]}:{t[2:4]}:{t[4:6]}'
    if iso:
        try:
            _d = datetime.fromisoformat(iso)
            date_str = _d.strftime('%Y-%m-%d')
            time_str = _d.strftime('%H:%M:%S')
        except Exception:
            pass
    return {
        'label':      label,
        'site':       site_name or '',
        'date':       date_str,
        'time':       time_str,
        'similarity_score': round(float(score), 8),
        'start_time': round(start_t, 6),
        'end_time':   round(end_t, 6),
        'recording_id': recording_id,
    }

def _search_one_db(db_path):
    meta = _load_db_metadata(db_path)
    emb_size = int(meta.get('embedding_size', 0))
    if emb_size and emb_size != _template_dim:
        print(f'  SKIP — DB embedding_size {emb_size} != template dim {_template_dim} '
              '(wrong backbone for this database).')
        return None
    sample_rate = int(float(meta.get('sample_rate', SAMPLE_RATE)))
    window_s    = float(meta.get('window_duration_s', WINDOW_S))
    overlap     = float(meta.get('overlap', 0.0))

    if MODE == 'per_label':
        heaps = {lbl: [] for lbl in _label_list}
    else:
        gheap = []
    counter = 0

    for keys, matrix in _iter_embeddings_batched(db_path, BATCH_SIZE):
        scores = _compute_scores(matrix, _flat_templates, METRIC)  # (B, T)
        B = len(keys)
        if MODE == 'per_label':
            for lbl, cols in _label_cols.items():
                lbl_scores = np.max(scores[:, cols], axis=1)
                heap = heaps[lbl]
                for bi in np.where(lbl_scores >= SCORE_THRESHOLD)[0]:
                    sc = float(lbl_scores[bi]); counter += 1
                    if len(heap) < TOP_K:
                        heapq.heappush(heap, (sc, counter, keys[int(bi)]))
                    elif sc > heap[0][0]:
                        heapq.heapreplace(heap, (sc, counter, keys[int(bi)]))
        else:
            all_scores = np.stack([np.max(scores[:, _label_cols[l]], axis=1) for l in _label_list])
            best_idx = np.argmax(all_scores, axis=0)
            best_sc  = all_scores[best_idx, np.arange(B)]
            for bi in np.where(best_sc >= SCORE_THRESHOLD)[0]:
                sc = float(best_sc[bi]); lbl = _label_list[int(best_idx[bi])]; counter += 1
                if len(gheap) < TOP_K:
                    heapq.heappush(gheap, (sc, counter, keys[int(bi)], lbl))
                elif sc > gheap[0][0]:
                    heapq.heapreplace(gheap, (sc, counter, keys[int(bi)], lbl))

    rows = []
    if MODE == 'per_label':
        for lbl, heap in heaps.items():
            for sc, _, key in sorted(heap, key=lambda x: -x[0]):
                rows.append(_row_from_key(key, lbl, sc, window_s, sample_rate, overlap))
        rows.sort(key=lambda r: (r['label'], -r['similarity_score']))
    else:
        for sc, _, key, lbl in sorted(gheap, key=lambda x: -x[0]):
            rows.append(_row_from_key(key, lbl, sc, window_s, sample_rate, overlap))
    return rows

# ── Discover databases ───────────────────────────────────────────────────────
if os.path.isdir(DRIVE_EMBEDDINGS_DB_PATH):
    _dbs = sorted(_glob.glob(os.path.join(DRIVE_EMBEDDINGS_DB_PATH, '**', '*.db'), recursive=True))
elif os.path.isfile(DRIVE_EMBEDDINGS_DB_PATH):
    _dbs = [DRIVE_EMBEDDINGS_DB_PATH]
else:
    raise FileNotFoundError(f'Databases path not found: {DRIVE_EMBEDDINGS_DB_PATH}')

if not _dbs:
    raise FileNotFoundError(f'No .db files found at: {DRIVE_EMBEDDINGS_DB_PATH}')

print(f'\nDatabases to search: {len(_dbs)}')
print('=' * 65)

_total_rows = 0
for _db in _dbs:
    _name = os.path.basename(_db)
    print(f'\n{_name}')
    _rows = _search_one_db(_db)
    if _rows is None:
        continue
    _stem = os.path.basename(_db)
    for _suf in ('.embeddings.db', '.db'):
        if _stem.endswith(_suf):
            _stem = _stem[:-len(_suf)]
            break
    _out = os.path.join(DRIVE_RESULTS_DIR, f'{_stem}.search.csv')
    with open(_out, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=_RESULT_COLS)
        w.writeheader()
        w.writerows(_rows)
    _total_rows += len(_rows)
    print(f'  → {len(_rows)} match(es) written to {_out}')

print('\n' + '=' * 65)
print(f'Vector search complete — {_total_rows} total match(es) across {len(_dbs)} database(s).')
print(f'Results folder: {DRIVE_RESULTS_DIR}')
print('\nContinue to Step 5 to extract the matching segments, then Step 6 to review them.')

---
## Step 5 — Extract Audio Segments

This step cuts a short audio clip for each match found in Step 4 and saves it to your Google Drive,
organised into one subfolder per site and label.

**How it works:**
- Each match stores the recording site, date, local time, and the exact position (in seconds) within the recording.
- The notebook locates the original recording in your Google Drive folder (or downloads it from Arbimon),
  cuts out the matching window, and saves it as a `.wav` file.

**Before running:** complete Step 4 so the result CSVs exist. Then run the cells below top to bottom.

> **Tip:** use the *species* and *max segments per label* settings below to limit how many clips are extracted.

In [57]:
#@title 📋 Load search results { display-mode: "form" }
#@markdown Reads the result CSV files written in Step 4 and prepares them for extraction.
#@markdown
#@markdown **Minimum score** — drop matches below this similarity score before extracting (0.0 keeps all).
MIN_SCORE = 0.7  #@param {type:"number"}

import os, glob, sqlite3
import pandas as pd

_csvs = sorted(glob.glob(os.path.join(DRIVE_RESULTS_DIR, '**', '*.search.csv'), recursive=True))
if not _csvs:
    raise FileNotFoundError(
        f'No "*.search.csv" files found in {DRIVE_RESULTS_DIR}. Run Step 4 first.')

_frames = [pd.read_csv(c) for c in _csvs]
df_raw = pd.concat(_frames, ignore_index=True)

# Map the search columns to the internal standard names used by Steps 5/6.
df = df_raw.rename(columns={
    'label':      'Common name',
    'site':       'Point name',
    'date':       'Date',
    'time':       'Time',
    'similarity_score': 'similarity_score',
})

df['Date']       = pd.to_datetime(df['Date'], errors='coerce').dt.date
df['Time']       = pd.to_datetime(df['Time'], format='%H:%M:%S', errors='coerce').dt.time
df['similarity_score'] = pd.to_numeric(df['similarity_score'], errors='coerce')
df['start_time'] = pd.to_numeric(df['start_time'], errors='coerce')
df['end_time']   = pd.to_numeric(df['end_time'], errors='coerce')

if MIN_SCORE > 0:
    df = df[df['similarity_score'] >= MIN_SCORE]

# Enrich with stream_id / utc_offset from the databases' `sites` tables so that
# extraction from Arbimon can convert local time back to UTC. Harmless for Drive.
_site_info = {}  # site_name -> (stream_id, utc_offset_int)
if os.path.isdir(DRIVE_EMBEDDINGS_DB_PATH):
    _db_files = glob.glob(os.path.join(DRIVE_EMBEDDINGS_DB_PATH, '**', '*.db'), recursive=True)
elif os.path.isfile(DRIVE_EMBEDDINGS_DB_PATH):
    _db_files = [DRIVE_EMBEDDINGS_DB_PATH]
else:
    _db_files = []
for _db in _db_files:
    try:
        _con = sqlite3.connect(_db)
        for _sn, _sid, _off in _con.execute('SELECT site_name, stream_id, utc_offset FROM sites'):
            _site_info[_sn] = (_sid or '', int(round(_off or 0)))
        _con.close()
    except Exception:
        pass

df['stream_id']  = df['Point name'].map(lambda s: _site_info.get(s, ('', 0))[0])
df['utc_offset'] = df['Point name'].map(lambda s: f"UTC{_site_info.get(s, ('', 0))[1]:+d}")

print(f'Result files loaded : {len(_csvs)}')
print(f'Total matches       : {len(df)}')
if not df.empty:
    print(f'Labels              : {sorted(df["Common name"].dropna().unique().tolist())}')
    print(f'Sites               : {sorted(df["Point name"].dropna().unique().tolist())}')
    print(f'Score range         : {df["similarity_score"].min():.4f} -> {df["similarity_score"].max():.4f}')
else:
    print('No matches remain after the minimum-score filter.')

In [58]:
#@title ⚙️ Extraction settings { display-mode: "form" }

#@markdown **Audio source** — where are the original recordings stored?
AUDIO_SOURCE = "google_drive"  #@param ["google_drive", "arbimon"]

#@markdown ---
#@markdown Full path to the folder containing audio files on your Drive.
#@markdown Subfolders matching the site names are searched automatically.
DRIVE_AUDIO_DIR = "/content/drive/MyDrive/audio"  #@param {type:"string"}

#@markdown ---
#@markdown **Output folder** — where extracted clips will be saved on your Drive.
#@markdown Clips are organised into one subfolder per site, then per label.
DRIVE_SEGMENTS_DIR = "/content/drive/MyDrive/vector_search_segments"  #@param {type:"string"}

#@markdown ---
#@markdown **Labels to extract** — leave blank to extract all labels in the results.
#@markdown Separate names with commas.  Example: `BOAALB, PHYLUT`
EXTRACT_SPECIES = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Maximum segments per label** — limits how many clips are extracted for each label.
#@markdown Set to **0** to extract all matches (no limit).
MAX_SEGMENTS_PER_LABEL = 25  #@param {type:"integer"}

#@markdown **Random seed** — controls which matches are selected when the limit above is applied.
#@markdown Use any positive integer for reproducible results, or **-1** for a different random draw each time.
RANDOM_SEED = -1  #@param {type:"integer"}

#@markdown ---
#@markdown **Padding (seconds)** — extra audio added before and after each matched window.
SEGMENT_PADDING_S = 0.0  #@param {type:"number"}

#@markdown **Output sample rate (Hz)** — rate for the saved WAV files.
#@markdown Use the same rate as your model (e.g. 48000 for BirdNET, 32000 for Google Perch).
EXTRACT_SR = 48000  #@param {type:"integer"}

import os
os.makedirs(DRIVE_SEGMENTS_DIR, exist_ok=True)
print(f'Audio source          : {AUDIO_SOURCE}')
print(f'Output folder         : {DRIVE_SEGMENTS_DIR}')
print(f'Max segments / label  : {"all" if MAX_SEGMENTS_PER_LABEL == 0 else MAX_SEGMENTS_PER_LABEL}')
print(f'Random seed           : {"off (random)" if RANDOM_SEED < 0 else RANDOM_SEED}')
print(f'Padding               : {SEGMENT_PADDING_S} s')
print(f'Sample rate           : {EXTRACT_SR} Hz')
if EXTRACT_SPECIES.strip():
    print(f'Label filter          : {[x.strip() for x in EXTRACT_SPECIES.split(",") if x.strip()]}')
else:
    print('Label filter          : all (from the loaded results)')

In [59]:
#@title 🔑 Log in to Arbimon (skip if using Google Drive) { display-mode: "form" }
#@markdown Run this cell only if your audio source is **Arbimon**.
CREDENTIALS_PATH = '/content/drive/MyDrive/rfcx/.rfcx_credentials'  #@param {type:"string"}

import os
if AUDIO_SOURCE == 'arbimon':
    # Install the Arbimon SDK here (not at the top) so it is only downloaded
    # when you actually use Arbimon as the audio source.
    !wget -q https://github.com/rfcx/rfcx-sdk-python/releases/download/0.3.1/rfcx-0.3.1-py3-none-any.whl -O /tmp/rfcx-0.3.1-py3-none-any.whl
    !pip install /tmp/rfcx-0.3.1-py3-none-any.whl -q
    import rfcx
    client_extract = rfcx.Client()
    if os.path.exists(CREDENTIALS_PATH):
        print('Credentials file found — logging in automatically...')
        client_extract.authenticate(persisted_credentials_path=CREDENTIALS_PATH)
    else:
        print('No saved credentials found.')
        print('A URL will appear below — open it in your browser and log in with your Arbimon account.')
        print('-' * 60)
        client_extract.authenticate(persisted_credentials_path=CREDENTIALS_PATH)
    print('\nLogged in to Arbimon successfully.')
else:
    print('Audio source is Google Drive — Arbimon login not needed. You can skip this cell.')

In [60]:
#@title ⬇️ Find or download recordings { display-mode: "form" }
#@markdown Identifies which original recordings contain the selected matches,
#@markdown then downloads them from Arbimon or locates them in your Drive folder.

import re, glob, datetime, unicodedata
import pandas as pd

# Build extraction subset
df_ext = df.copy()
if EXTRACT_SPECIES.strip():
    _ext_sp = [x.strip() for x in EXTRACT_SPECIES.split(',') if x.strip()]
    df_ext  = df_ext[df_ext['Common name'].isin(_ext_sp)]

if df_ext.empty:
    raise ValueError(
        'No matches meet the extraction criteria.\n'
        'Check your label list and the minimum score in the cells above.'
    )

# ── Random sampling per label ─────────────────────────────────────────────────
if MAX_SEGMENTS_PER_LABEL > 0:
    _seed = RANDOM_SEED if RANDOM_SEED >= 0 else None
    df_ext = (
        df_ext
        .groupby('Common name', group_keys=False)
        .apply(lambda g: g.sample(n=min(len(g), MAX_SEGMENTS_PER_LABEL), random_state=_seed))
    )
    df_ext = df_ext.sort_index()
    print(f'Sampling applied      : up to {MAX_SEGMENTS_PER_LABEL} per label'
          f'  (seed={_seed if _seed is not None else "random"})')
    for sp, grp in df_ext.groupby('Common name'):
        print(f'  {sp}: {len(grp)} match(es) selected')
else:
    print('Sampling              : disabled (extracting all matches)')

print(f'\nMatches to extract       : {len(df_ext)}')

def _utc_hours(utc_str):
    try:
        return int(str(utc_str).replace('UTC', '').strip() or 0)
    except ValueError:
        return 0

def _rec_datetime(row):
    # Returns the datetime of the recording start that contains this match.
    # Used only to locate the audio file — start_time/end_time are offsets from
    # the start of that file (0 = beginning of the recording).
    # For Arbimon: converts local time to UTC using utc_offset.
    # For Google Drive: keeps local time (filenames use local time).
    try:
        local_dt = datetime.datetime.combine(row['Date'], row['Time'])
        if AUDIO_SOURCE == 'arbimon':
            local_dt = local_dt - datetime.timedelta(hours=_utc_hours(row.get('utc_offset', 'UTC+0')))
        return local_dt
    except Exception:
        return None

df_ext = df_ext.copy()
df_ext['_rec_dt'] = df_ext.apply(_rec_datetime, axis=1)

if AUDIO_SOURCE == 'arbimon':
    unique_recs = df_ext[['stream_id', '_rec_dt']].dropna().drop_duplicates()
else:
    # Keep the site: two sites often record at the same scheduled time, so the
    # recording is only uniquely identified by (site, datetime).
    unique_recs = df_ext[['Point name', '_rec_dt']].dropna().drop_duplicates()

print(f'Unique recordings needed : {len(unique_recs)}')

AUDIO_TEMP_DIR = '/content/audio_extract'
os.makedirs(AUDIO_TEMP_DIR, exist_ok=True)

audio_file_map = {}  # key → local audio file path

# ── Arbimon: one download call per needed recording ──────────────────────────
if AUDIO_SOURCE == 'arbimon':
    dest = AUDIO_TEMP_DIR
    os.makedirs(dest, exist_ok=True)

    def _arbimon_file_dt(filepath):
        name = os.path.basename(filepath)
        m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
        if m:
            return datetime.datetime(*[int(g) for g in m.groups()])
        return None

    for _, rec in unique_recs.iterrows():
        sid = str(rec['stream_id'])
        t0  = rec['_rec_dt']
        key = (sid, t0)

        existing = (glob.glob(f'{dest}/**/*.wav',  recursive=True) +
                    glob.glob(f'{dest}/**/*.flac', recursive=True))
        already = next(
            (f for f in existing
             if sid in os.path.basename(f)
             and (fdt := _arbimon_file_dt(f)) is not None
             and abs((fdt - t0).total_seconds()) < 2),
            None
        )
        if already:
            audio_file_map[key] = already
            print(f'  Cached  : {os.path.basename(already)}')
            continue

        q_min = t0 - datetime.timedelta(seconds=5)
        q_max = t0 + datetime.timedelta(seconds=55)
        try:
            client_extract.download_segments(
                dest_path=dest, stream=sid,
                min_date=q_min, max_date=q_max, parallel=False,
            )
        except Exception as e:
            print(f'  ERROR — stream {sid} at {t0}: {e}')
            continue

        all_downloaded = (glob.glob(f'{dest}/**/*.wav',  recursive=True) +
                          glob.glob(f'{dest}/**/*.flac', recursive=True))
        matched = next(
            (f for f in all_downloaded
             if sid in os.path.basename(f)
             and (fdt := _arbimon_file_dt(f)) is not None
             and abs((fdt - t0).total_seconds()) < 2),
            None
        )
        if matched:
            audio_file_map[key] = matched
            print(f'  Downloaded : {os.path.basename(matched)}')
        else:
            print(f'  WARNING: no file found for stream {sid} at {t0}')

# ── Google Drive: match files by datetime in filename, within the site folder ─
elif AUDIO_SOURCE == 'google_drive':
    all_audio = (
        glob.glob(f'{DRIVE_AUDIO_DIR}/**/*.wav',  recursive=True) +
        glob.glob(f'{DRIVE_AUDIO_DIR}/**/*.flac', recursive=True) +
        glob.glob(f'{DRIVE_AUDIO_DIR}/**/*.mp3',  recursive=True)
    )
    print(f'Audio files on Drive : {len(all_audio)}')

    def _file_dt(filepath):
        name = os.path.basename(filepath)
        m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
        if m:
            return datetime.datetime(*[int(g) for g in m.groups()])
        m = re.search(r'(\d{8})_(\d{6})', name)
        if m:
            d, t = m.group(1), m.group(2)
            return datetime.datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                                     int(t[:2]), int(t[2:4]), int(t[4:6]))
        return None

    def _norm(s):
        # Normalise for comparison: strip accents (NFKD → drop combining marks →
        # ASCII), lowercase, keep only letters/digits. This makes 'POÇA' match
        # regardless of how the cedilla is encoded (NFC vs NFD), while still
        # keeping 'Ponto 1' distinct from 'Ponto 10'. Accent-stripping is the fix
        # for accented site names (POÇA, ÁGUA, SÃO, ...).
        s = unicodedata.normalize('NFKD', str(s))
        s = ''.join(ch for ch in s if not unicodedata.combining(ch))
        s = s.encode('ascii', 'ignore').decode('ascii')
        return re.sub(r'[^a-z0-9]', '', s.lower())

    def _file_site_match(filepath, site):
        # Per-site subfolders: the site must be one of the FOLDER names below
        # DRIVE_AUDIO_DIR (the filename is ignored here).
        rel     = os.path.relpath(filepath, DRIVE_AUDIO_DIR)
        folders = [_norm(p) for p in rel.split(os.sep)[:-1]]
        return _norm(site) in folders

    _multi_site = unique_recs['Point name'].nunique() > 1
    n_unscoped = 0   # site folder not found
    n_notime   = 0   # no file at the expected time

    for _, rec in unique_recs.iterrows():
        site = str(rec['Point name'])
        t0   = rec['_rec_dt']
        key  = (site, t0)

        # Restrict to this site's subfolder so two sites recorded at the same
        # scheduled time can never be confused.
        site_files = [f for f in all_audio if _file_site_match(f, site)]

        if site_files:
            search = site_files
        elif _multi_site:
            # Fail loudly instead of guessing. With several sites, matching by
            # time alone can silently grab another site's recording (recorders
            # are usually synchronised, so every site has a file at this exact
            # time). A missing clip is noticed; a wrong clip is not.
            print(f'  ERROR: no subfolder matched site "{site}" — skipping this '
                  f'recording. Rename a subfolder under the audio path to match the '
                  f'"site" value ("{site}"), then re-run.')
            n_unscoped += 1
            continue
        else:
            # Single site: there is no other site to confuse it with.
            search = all_audio

        time_hits = sorted(
            f for f in search
            if (fdt := _file_dt(f)) and abs((fdt - t0).total_seconds()) < 5
        )
        if not time_hits:
            print(f'  WARNING: no audio file for site "{site}" at {t0} — skipping.')
            n_notime += 1
            continue

        matched = time_hits[0]
        audio_file_map[key] = matched
        if len(time_hits) > 1:
            print(f'  WARNING: {len(time_hits)} files match site "{site}" at {t0}; '
                  f'using {os.path.basename(matched)}.')
        else:
            print(f'  Matched : {site} -> {os.path.basename(matched)}')

    if n_unscoped:
        print(f'\n{n_unscoped} recording(s) skipped — site folder not found. '
              f'These matches will NOT be extracted until the folder names '
              f'match the "site" column.')
    if n_notime:
        print(f'{n_notime} recording(s) skipped — no audio file at the expected time.')

print(f'\nAudio files ready : {len(audio_file_map)} / {len(unique_recs)}')

In [61]:
#@title ✂️ Extract and save audio segments { display-mode: "form" }
#@markdown Cuts each match out of its recording and saves it as a WAV file.
#@markdown Clips are saved into one subfolder per site, then per label, inside the output folder.
#@markdown Detection times are offsets from the start of the recording (0 = beginning),
#@markdown so the clip is cut at start_time..end_time within the matched file.
#@markdown
#@markdown File names follow the pattern:
#@markdown `SITE_YYYYMMDD_HHMMSS_START-ENDs_SCORE_LABEL.wav`

import os
import librosa, soundfile as sf
import numpy as np

saved   = 0
skipped = 0
errors  = 0

for _, row in df_ext.iterrows():
    if AUDIO_SOURCE == 'arbimon':
        key = (str(row['stream_id']), row['_rec_dt'])
    else:
        key = (str(row['Point name']), row['_rec_dt'])
    audio_path = audio_file_map.get(key)

    if not audio_path:
        errors += 1
        continue

    # start_time/end_time are seconds from the start of the recording (0-based).
    det_start  = float(row['start_time'])
    det_end    = float(row['end_time'])
    start_s    = max(0.0, det_start - SEGMENT_PADDING_S)
    end_s      = det_end + SEGMENT_PADDING_S
    conf       = float(row['similarity_score'])

    site_safe  = str(row['Point name']).replace(' ', '_')
    label_safe = str(row['Common name']).replace(' ', '_').replace('/', '-')
    date_str   = str(row['Date']).replace('-', '')
    time_str   = str(row['Time']).replace(':', '')[:6]
    conf_str   = f'{conf:.3f}'
    # Keep the fractional detection start in the name so two matches in the
    # same second are not collapsed onto one filename (and silently skipped).
    start_tag  = f'{det_start:.1f}'
    end_tag    = f'{det_end:.1f}'
    out_name   = f'{site_safe}_{date_str}_{time_str}_{start_tag}-{end_tag}s_{conf_str}_{label_safe}.wav'

    seg_out = os.path.join(DRIVE_SEGMENTS_DIR, site_safe, label_safe)
    os.makedirs(seg_out, exist_ok=True)
    out_path = os.path.join(seg_out, out_name)

    if os.path.exists(out_path):
        skipped += 1
        continue

    try:
        audio, _ = librosa.load(audio_path, sr=EXTRACT_SR, mono=True,
                                offset=start_s, duration=end_s - start_s)
    except Exception as e:
        print(f'  ERROR loading {os.path.basename(audio_path)}: {e}')
        errors += 1
        continue

    sf.write(out_path, audio, EXTRACT_SR)
    saved += 1

print(f'Segments saved     : {saved}')
if skipped:
    print(f'Already existed    : {skipped} (skipped)')
if errors:
    print(f'Skipped (no audio) : {errors}')
print(f'Output folder      : {DRIVE_SEGMENTS_DIR}')

---
## Step 6 — Review Extracted Segments

Browse the audio clips extracted in Step 5 one at a time.
Each segment is shown as a spectrogram with its label and similarity score.
You can listen to it with the built-in audio player, then mark it as **true** (a genuine match) or **false** (a false positive).

- Marking a segment moves its file into a `true/` or `false/` subfolder inside the segments folder.
- Use **← Prev** and **Next →** to browse without making a decision.
- Already-reviewed files (inside `true/` or `false/`) are excluded from the list automatically each time you run the cell.
- The **spectrogram controls** (type, frequency range, dB floor) update the view instantly without reloading the audio.

> Run Step 5 first so the segments folder is populated, then run the two cells below.

In [62]:
#@title ⚙️ Review settings { display-mode: "form" }
#@markdown Full path to the folder containing the extracted segments.
#@markdown Use the same value as **DRIVE_SEGMENTS_DIR** from Step 5.
REVIEW_DIR = "/content/drive/MyDrive/vector_search_segments"  #@param {type:"string"}

#@markdown ---
#@markdown ### Spectrogram defaults
#@markdown These set the initial values shown in the reviewer.
#@markdown You can adjust all of them live inside the GUI without re-running this cell.

#@markdown **Spectrogram type** — `mel` uses a perceptual frequency scale; `fft` uses a linear Hz scale.
SPEC_TYPE = "mel"  #@param ["mel", "fft"]

#@markdown **Minimum frequency (Hz)** — lower bound of the frequency axis.
FREQ_MIN_HZ = 0  #@param {type:"integer"}

#@markdown **Maximum frequency (Hz)** — upper bound. Set to **0** to use the Nyquist frequency (half the sample rate).
FREQ_MAX_HZ = 0  #@param {type:"integer"}

#@markdown **dB floor** — lowest value shown on the colour scale (e.g. −80). Typical range: −100 to −40.
DB_MIN = -80  #@param {type:"integer"}

import os
os.makedirs(os.path.join(REVIEW_DIR, 'true'),  exist_ok=True)
os.makedirs(os.path.join(REVIEW_DIR, 'false'), exist_ok=True)

import glob as _g
_all  = _g.glob(f'{REVIEW_DIR}/**/*.wav', recursive=True)
_skip = {os.path.join(REVIEW_DIR, 'true'), os.path.join(REVIEW_DIR, 'false')}
_pending = [f for f in _all
            if not any(os.path.commonpath([f, s]) == s for s in _skip)]

print(f'Segments folder  : {REVIEW_DIR}')
print(f'Pending review   : {len(_pending)} segment(s)')
print(f'Already true     : {len(_g.glob(os.path.join(REVIEW_DIR, "true",  "*.wav")))}')
print(f'Already false    : {len(_g.glob(os.path.join(REVIEW_DIR, "false", "*.wav")))}')
print(f'Spectrogram type : {SPEC_TYPE}')
print(f'Freq range       : {FREQ_MIN_HZ} – {"Nyquist" if FREQ_MAX_HZ == 0 else str(FREQ_MAX_HZ) + " Hz"}')
print(f'dB floor         : {DB_MIN} dB')

In [63]:
#@title 🎧 Segment reviewer { display-mode: "form" }
#@markdown Run this cell to open the interactive reviewer.
#@markdown Use **True / False** to classify a segment and move to the next one.
#@markdown Use **← Prev / Next →** to browse without classifying.
#@markdown Adjust the spectrogram controls to change the view — the audio player is unaffected.
#@markdown When **False** is pressed, enter the correct label before confirming.

import os, glob, shutil, re, io, base64
import numpy as np
from datetime import datetime
import librosa
import librosa.display
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# 1x1 transparent PNG — assigning b'' does not reliably clear an Image
# widget in Colab, which can leave the previous segment's spectrogram on
# screen next to the new audio. Use this as an explicit 'blank'.
_BLANK_PNG = base64.b64decode(
    'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNk+M9QDwADhgGAWjR9awAAAABJRU5ErkJggg=='
)

# ── Collect unreviewed segments ───────────────────────────────────────────────
def _collect(base):
    skip = {os.path.join(base, 'true'), os.path.join(base, 'false')}
    return sorted(
        f for f in glob.glob(f'{base}/**/*.wav', recursive=True)
        if not any(os.path.commonpath([f, s]) == s for s in skip)
    )

# ── Parse label + score from filename ─────────────────────────────────────────
def _parse(filepath):
    stem = os.path.splitext(os.path.basename(filepath))[0]
    # Similarity score is the last _<number>_ token before the label; the greedy
    # prefix forces the match onto it even if other decimals appear earlier.
    m = re.search(r'.*_(\d+\.\d+)_(.+)$', stem)
    label = m.group(2).replace('_', ' ') if m else stem
    conf  = float(m.group(1)) if m else None
    # Filenames follow SITE_YYYYMMDD_HHMMSS_START-ENDs_CONF_LABEL.wav; the site
    # itself may contain underscores, so anchor on the fixed date/time tokens
    # instead of splitting blindly.
    site, dt = None, None
    m2 = re.match(r'(.+)_(\d{8})_(\d{6})_[\d.]+-[\d.]+s_', stem)
    if m2:
        site = m2.group(1).replace('_', ' ')
        try:
            dt = datetime.strptime(m2.group(2) + m2.group(3), '%Y%m%d%H%M%S')
        except ValueError:
            dt = None
    return label, conf, site, dt

# ── State ─────────────────────────────────────────────────────────────────────
_state = {'idx': 0, 'segs': _collect(REVIEW_DIR)}

# ── Spectrogram controls ──────────────────────────────────────────────────────
_W_ctrl = {'style': {'description_width': 'initial'}}

_dd_type = widgets.Dropdown(
    options=[('Mel', 'mel'), ('FFT (linear)', 'fft')],
    value=globals().get('SPEC_TYPE', 'mel'),
    description='Type:',
    layout=widgets.Layout(width='165px'),
    **_W_ctrl,
)
_int_fmin = widgets.BoundedIntText(
    value=globals().get('FREQ_MIN_HZ', 0), min=0, max=96000, step=100,
    description='Min Hz:',
    layout=widgets.Layout(width='155px'),
    **_W_ctrl,
)
_int_fmax = widgets.BoundedIntText(
    value=globals().get('FREQ_MAX_HZ', 0), min=0, max=96000, step=100,
    description='Max Hz:',
    layout=widgets.Layout(width='155px'),
    **_W_ctrl,
)
_int_db = widgets.BoundedIntText(
    value=globals().get('DB_MIN', -80), min=-120, max=-20, step=5,
    description='dB floor:',
    layout=widgets.Layout(width='160px'),
    **_W_ctrl,
)
_ctrl_row = widgets.HBox(
    [_dd_type, _int_fmin, _int_fmax, _int_db,
     widgets.Label('(Max Hz = 0 → Nyquist)', layout=widgets.Layout(margin='6px 0 0 4px'))],
    layout=widgets.Layout(justify_content='center', margin='4px 0', flex_wrap='wrap'),
)

# ── Display widgets ───────────────────────────────────────────────────────────
_spec_widget  = widgets.Image(value=b'', format='png',
                               layout=widgets.Layout(width='720px', height='auto'))
_audio_widget = widgets.HTML(value='')
_lbl_prog     = widgets.HTML(value='')
_lbl_error    = widgets.HTML(value='')

# ── Navigation row ────────────────────────────────────────────────────────────
_BW = widgets.Layout(width='120px', height='36px')
_btn_prev  = widgets.Button(description='← Prev',   layout=widgets.Layout(width='100px', height='36px'))
_btn_true  = widgets.Button(description='✔  True',  button_style='success', layout=_BW)
_btn_false = widgets.Button(description='✘  False', button_style='danger',  layout=_BW)
_btn_next  = widgets.Button(description='Next →',   layout=widgets.Layout(width='100px', height='36px'))

_nav_row = widgets.HBox(
    [_btn_prev, _btn_true, _btn_false, _btn_next],
    layout=widgets.Layout(justify_content='center', margin='8px 0'),
)

# ── False-label input row (hidden until False is pressed) ─────────────────────
_input_label       = widgets.Text(
    placeholder='Correct label  (leave blank → "unknown")',
    layout=widgets.Layout(width='340px', height='36px'),
)
_btn_confirm_false = widgets.Button(description='Confirm', button_style='warning',
                                     layout=widgets.Layout(width='100px', height='36px'))
_btn_cancel        = widgets.Button(description='Cancel',
                                     layout=widgets.Layout(width='80px',  height='36px'))

_label_row = widgets.VBox([
    widgets.HTML('<span style="font-size:12px">Enter the correct label for this segment:</span>'),
    widgets.HBox(
        [_input_label, _btn_confirm_false, _btn_cancel],
        layout=widgets.Layout(justify_content='center', margin='4px 0'),
    ),
], layout=widgets.Layout(align_items='center', margin='4px 0', display='none'))

# ── Full UI ───────────────────────────────────────────────────────────────────
_ui = widgets.VBox([
    _lbl_prog,
    _ctrl_row,
    _spec_widget,
    _audio_widget,
    _lbl_error,
    _label_row,
    _nav_row,
], layout=widgets.Layout(align_items='center'))

# ── Spectrogram rendering ─────────────────────────────────────────────────────
def _show_spec():
    segs = _state['segs']
    if not segs:
        _spec_widget.value = _BLANK_PNG
        return
    filepath = segs[_state['idx']]
    label, conf, site, dt = _parse(filepath)
    conf_str  = f'{conf:.3f}' if conf is not None else '?'
    site_str  = site if site else '?'
    dt_str    = dt.strftime('%Y-%m-%d %H:%M:%S') if dt else '?'
    spec_type = _dd_type.value
    fmin_hz   = _int_fmin.value
    db_floor  = _int_db.value
    _lbl_error.value = ''
    try:
        y, sr   = librosa.load(filepath, sr=None, mono=True)
        fmax_hz = _int_fmax.value if _int_fmax.value > 0 else sr // 2
        fig, ax = plt.subplots(figsize=(9, 4))
        if spec_type == 'mel':
            S   = librosa.feature.melspectrogram(
                y=y, sr=sr, n_mels=128, fmin=fmin_hz, fmax=fmax_hz)
            Sd  = librosa.power_to_db(S, ref=np.max)
            img = librosa.display.specshow(
                Sd, sr=sr, x_axis='time', y_axis='mel', ax=ax,
                fmin=fmin_hz, fmax=fmax_hz, vmin=db_floor, vmax=0)
        else:
            D   = librosa.stft(y)
            Sd  = librosa.amplitude_to_db(np.abs(D), ref=np.max)
            img = librosa.display.specshow(
                Sd, sr=sr, x_axis='time', y_axis='hz', ax=ax,
                vmin=db_floor, vmax=0)
            ax.set_ylim(fmin_hz, fmax_hz)
        ax.set_title(f'{label}   |   score: {conf_str}   |   {site_str}   |   {dt_str}', fontsize=12, fontweight='bold')
        fig.colorbar(img, ax=ax, format='%+2.0f dB')
        plt.tight_layout()
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        _spec_widget.value = buf.read()
    except Exception as e:
        plt.close('all')
        _spec_widget.value = _BLANK_PNG
        _lbl_error.value = f'<span style="color:red">Spectrogram error: {e}</span>'

def _show_audio():
    segs = _state['segs']
    if not segs:
        _audio_widget.value = ''
        return
    try:
        with open(segs[_state['idx']], 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        _audio_widget.value = (
            f'<audio controls style="width:720px;margin-top:4px">'
            f'<source src="data:audio/wav;base64,{b64}" type="audio/wav">'
            f'</audio>'
        )
    except Exception as e:
        _audio_widget.value = f'<span style="color:red">Audio error: {e}</span>'

def _show():
    segs = _state['segs']
    n    = len(segs)
    if n == 0:
        _lbl_prog.value     = '<b>All segments have been reviewed.</b>'
        _spec_widget.value  = _BLANK_PNG
        _audio_widget.value = ''
        return
    _lbl_prog.value = (
        f'<span style="font-size:13px">Segment <b>{_state["idx"] + 1}</b> of <b>{n}</b></span>'
    )
    _show_spec()
    _show_audio()

# ── File operations ───────────────────────────────────────────────────────────
def _move(verdict, subfolder=None):
    segs = _state['segs']
    if not segs:
        return
    dest_dir = os.path.join(REVIEW_DIR, verdict, subfolder) if subfolder \
               else os.path.join(REVIEW_DIR, verdict)
    os.makedirs(dest_dir, exist_ok=True)
    src = segs.pop(_state['idx'])
    shutil.move(src, os.path.join(dest_dir, os.path.basename(src)))
    if _state['idx'] >= len(segs) and _state['idx'] > 0:
        _state['idx'] -= 1
    _show()

def _nav(delta):
    n = len(_state['segs'])
    if n == 0:
        return
    _state['idx'] = max(0, min(n - 1, _state['idx'] + delta))
    _show()

# ── Button handlers ───────────────────────────────────────────────────────────
def _show_label_input(_):
    _input_label.value        = ''
    _label_row.layout.display = ''
    _nav_row.layout.display   = 'none'

def _on_confirm_false(_):
    raw      = _input_label.value.strip()
    subfolder = raw.replace(' ', '_').replace('/', '-') if raw else 'unknown'
    _label_row.layout.display = 'none'
    _nav_row.layout.display   = ''
    _move('false', subfolder)

def _on_cancel(_):
    _label_row.layout.display = 'none'
    _nav_row.layout.display   = ''

_btn_true.on_click(lambda _: _move('true'))
_btn_false.on_click(_show_label_input)
_btn_confirm_false.on_click(_on_confirm_false)
_btn_cancel.on_click(_on_cancel)
_btn_prev.on_click(lambda _: _nav(-1))
_btn_next.on_click(lambda _: _nav(+1))

# ── Spec-change observers ─────────────────────────────────────────────────────
def _on_spec_change(change):
    if _state['segs']:
        _show_spec()

for _ctrl in (_dd_type, _int_fmin, _int_fmax, _int_db):
    _ctrl.observe(_on_spec_change, names='value')

# ── Launch ────────────────────────────────────────────────────────────────────
_show()
display(_ui)